# Explore single home
This calculates cycling features for a single home and does some visualisations

In [39]:
import pandas as pd
import numpy as np
import plotly.express as px
import cycling_fns as cf

In [40]:
# Load in the summary data and filter to ASHPs that were included in the analysis
summary_data = pd.read_csv("DESNZ Electrification of Heat Project - Heat Pump Performance Data Summary.csv")
heat_pumps = summary_data.loc[summary_data["Included_SPF_analysis"], ["Property_ID", "HP_Installed", "HP_Brand", "HP_Model", "HP_Size_kW", "HP_Refrigerant", "SPFH4_selected_window",
                                                                      "HP_Energy_Output_selected_window", "Mean_annual_SH_flow_temp_selected_window", 'Mean_annual_HW_flow_temp_selected_window',
                                                                      'Selected_window_start', 'Selected_window_end']]
heat_pumps = heat_pumps[heat_pumps["HP_Installed"].isin(["ASHP", "HT_ASHP"])]

In [ ]:
# "EOH2232" (cycling temp drop example), "EOH1893" (clear weather comp) "EOH2974" (fixed flow temp) #EOH0311" (two-step fixed weather comp) # "EOH1154" (cycling graph)
property_id = "EOH2232" 
readings = pd.read_csv(f"../../../monitoring_analysis_2/notebooks/clean/Property_ID={property_id}.csv")

In [42]:
readings = cf.prep_readings(readings)
readings = cf.flag_cycle_groups(readings)
max_power = cf.calc_max_power(readings)
cycles = cf.identify_cycles(readings)
cycling_features = cf.calc_cycling_features(cycles, max_power)

In [43]:
cycles

,state,start_time,duration,max_power,min_power,mean_power,std_power,max_heat_temp,min_heat_temp,median_heat_temp,mean_external_temp,mean_internal_temp,std_heat_temp,max_hot_water_temp,min_hot_water_temp,median_hot_water_temp,std_hot_water_temp,heat_temp_diff,hot_water,temp_change
0,0,2021-12-01 00:00:00+00:00,0.0,90.0,90.0,90.000000,NaN,NaN,NaN,NaN,10.400000,20.640000,NaN,33.57,33.57,33.57,NaN,NaN,True,NaN
1,1,2021-12-01 00:02:00+00:00,6.0,1770.0,690.0,1417.500000,495.000000,53.66,53.66,53.660,10.400000,20.640000,NaN,52.55,47.61,50.91,2.516055,NaN,True,0.00
2,0,2021-12-01 00:10:00+00:00,2.0,150.0,30.0,90.000000,84.852814,45.40,45.40,45.400,10.400000,20.640000,NaN,47.30,47.30,47.30,NaN,NaN,True,0.00
3,1,2021-12-01 00:14:00+00:00,6.0,1770.0,1170.0,1477.500000,292.275555,39.90,39.90,39.900,10.400000,20.640000,NaN,52.98,49.59,51.14,1.697066,NaN,True,0.00
4,0,2021-12-01 00:22:00+00:00,0.0,90.0,90.0,90.000000,NaN,NaN,NaN,NaN,10.400000,20.640000,NaN,48.42,48.42,48.42,NaN,NaN,True,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21553,1,2023-02-28 22:28:00+00:00,76.0,3330.0,870.0,2349.230769,507.259845,54.97,33.24,42.015,2.070513,21.619231,7.618292,55.91,37.51,52.49,5.366269,NaN,True,21.73
21554,0,2023-02-28 23:46:00+00:00,0.0,120.0,120.0,120.000000,NaN,NaN,NaN,NaN,1.190000,21.680000,NaN,49.91,49.91,49.91,NaN,NaN,True,NaN
21555,1,2023-02-28 23:48:00+00:00,6.0,2640.0,1740.0,2047.500000,402.771647,38.58,35.79,37.365,1.085000,21.680000,1.184019,NaN,NaN,NaN,NaN,1.59,False,2.79
21556,0,2023-02-28 23:56:00+00:00,0.0,150.0,150.0,150.000000,NaN,33.29,33.29,33.290,0.980000,21.680000,NaN,NaN,NaN,NaN,NaN,NaN,False,0.00


In [44]:
readings.columns

Index(['Circulation_Pump_Energy_Consumed', 'External_Air_Temperature',
       'Heat_Pump_Energy_Output', 'Heat_Pump_Heating_Flow_Temperature',
       'Heat_Pump_Return_Temperature', 'Hot_Water_Flow_Temperature',
       'Immersion_Heater_Energy_Consumed', 'Internal_Air_Temperature',
       'Whole_System_Energy_Consumed', 'Power_W', 'Heat_Pump_Power_Output',
       'on', 'group', 'position'],
      dtype='object')

In [45]:
import plotly.graph_objects as go

# Define the time range
df_filtered = readings.loc['2022-01-08 01:00:00':'2022-01-08 12:00:00'].copy()
df_filtered["comb_flow_temp"] = df_filtered["Heat_Pump_Heating_Flow_Temperature"].fillna(df_filtered["Hot_Water_Flow_Temperature"])

# Create the figure
fig = go.Figure()

# Add the first y-axis (Power values) as bars
fig.add_trace(go.Bar(
    x=df_filtered.index,
    y=df_filtered["Power_W"],
    name="Power (W)",  
    marker_color="black",
    opacity=0.7
))

# fig.add_trace(go.Bar(
#     x=df_filtered.index,
#     y=df_filtered["Heat_Pump_Power_Output"],
#     name="Heat Pump Power Output",
#     marker_color="red",
#     opacity=0.7
# ))

# Add the second y-axis (Flow Temperatures)
fig.add_trace(go.Scatter(
    x=df_filtered.index,
    y=df_filtered["comb_flow_temp"],  # Replace with the actual column name for flow temperature
    name="Flow Temperature (°C)",
    mode="lines",
    line=dict(color="green", width=1),
    yaxis="y2"  # Assign to second y-axis
))

# Add shaded areas where on == 1
for i in range(1, len(df_filtered)):
    if df_filtered['on'].iloc[i] == 1 and df_filtered['on'].iloc[i - 1] == 0:
        # Start of shaded region
        start_time = df_filtered.index[i-1]
    elif df_filtered['on'].iloc[i] == 0 and df_filtered['on'].iloc[i - 1] == 1:
        # End of shaded region
        end_time = df_filtered.index[i]

        if np.isnan(df_filtered["Hot_Water_Flow_Temperature"].iloc[i-1]):
            fill_colour="rgba(255, 0, 0, 0.3)"
        else:
            fill_colour="rgba(0, 0, 255, 0.3)"
        # Add shaded area
        fig.add_shape(
            type="rect",
            x0=start_time,
            x1=end_time,
            y0=0, y1=1,  # Extend full height
            xref="x",
            yref="paper",
            fillcolor=fill_colour,
            line_width=0
        )

fig.add_shape(
    type="line",
    x0=df_filtered.index.min(),
    x1=df_filtered.index.max(),
    y0=200,
    y1=200,
    line=dict(color="black", width=2, dash="dash"),  # Customize color, width, and dash style
)


# Update layout to include secondary y-axis
fig.update_layout(
    width=900,
    title="Detecting Heating and Hot Water Cycles",
    xaxis=dict(title="Time"),
    yaxis=dict(
        title="Heat Pump Electrical Power (W)",
        side="left",
        showgrid=False
    ),
    yaxis2=dict(
        title="Flow Temperature (°C)",
        overlaying="y",
        side="right",
        showgrid=False
    ),
    legend=dict(x=0, y=1)    
)

# Show the figure
fig.show()


In [46]:
readings['2022-01-08 04:50':'2022-01-08 05:20']

,Circulation_Pump_Energy_Consumed,External_Air_Temperature,Heat_Pump_Energy_Output,Heat_Pump_Heating_Flow_Temperature,Heat_Pump_Return_Temperature,Hot_Water_Flow_Temperature,Immersion_Heater_Energy_Consumed,Internal_Air_Temperature,Whole_System_Energy_Consumed,Power_W,Heat_Pump_Power_Output,on,group,position
2022-01-08 04:50:00+00:00,85.009,2.30,6733.304,54.80,48.06,NaN,168.238,23.13,2861.361,4650.0,10680.0,1,4432,13
2022-01-08 04:52:00+00:00,85.011,2.32,6733.644,54.35,48.33,NaN,168.238,23.25,2861.502,4230.0,10200.0,1,4432,14
2022-01-08 04:54:00+00:00,85.012,2.34,6733.976,54.31,48.53,NaN,168.238,23.25,2861.633,3930.0,9960.0,1,4432,15
2022-01-08 04:56:00+00:00,85.014,2.36,6734.279,54.32,48.64,NaN,168.238,23.37,2861.766,3990.0,9090.0,1,4432,16
2022-01-08 04:58:00+00:00,85.016,2.38,6734.576,54.33,48.70,NaN,168.238,23.37,2861.898,3960.0,8910.0,1,4432,17
2022-01-08 05:00:00+00:00,85.018,2.40,6734.835,NaN,47.06,49.73,168.238,23.37,2861.965,2010.0,7770.0,1,4432,18
2022-01-08 05:02:00+00:00,85.020,2.47,6734.869,NaN,46.26,45.83,168.238,23.47,2861.968,90.0,1020.0,0,4433,1
2022-01-08 05:04:00+00:00,85.022,2.54,6734.869,NaN,47.02,46.68,168.238,23.47,2861.971,90.0,0.0,0,4433,2
2022-01-08 05:06:00+00:00,85.023,2.61,6734.869,NaN,46.85,46.76,168.238,23.47,2861.974,90.0,0.0,0,4433,3
2022-01-08 05:08:00+00:00,85.025,2.68,6734.869,NaN,46.70,46.62,168.238,23.47,2861.976,60.0,0.0,0,4433,4


In [47]:
px.scatter(cycles[(cycles["state"]==1) & ~cycles["hot_water"]], x="mean_external_temp", y="max_heat_temp", 
           labels={
                "mean_external_temp": "External Temperature (C)",
                "max_heat_temp": "Maximum Heating Flow Temperature (C)",
            }, width=600)

In [48]:
fig = px.scatter(cycles[(cycles["state"]==1) & ~cycles["hot_water"]], x="mean_external_temp", y="median_heat_temp", 
           title=f"Empirical weather compensation curve for {property_id}",
           labels={
                "mean_external_temp": "External Temperature (C)",
                "median_heat_temp": "Median Heating Flow Temperature (C)",
            },           
           width=600,
           trendline="ols")
# Change trendline color
#fig.update_traces(selector=dict(name="OLS Trendline"), line=dict(color="black", width=3))

# Force update of the trendline color
fig.data[-1].line.color = "black"  # Trendline is usually the last trace
fig.data[-1].line.width = 3  # Make it more visible
fig.show()

In [49]:
px.scatter(cycles[(cycles["state"]==1) & ~cycles["hot_water"]], x="start_time", y="median_heat_temp", width=600)

In [50]:
readings.columns

Index(['Circulation_Pump_Energy_Consumed', 'External_Air_Temperature',
       'Heat_Pump_Energy_Output', 'Heat_Pump_Heating_Flow_Temperature',
       'Heat_Pump_Return_Temperature', 'Hot_Water_Flow_Temperature',
       'Immersion_Heater_Energy_Consumed', 'Internal_Air_Temperature',
       'Whole_System_Energy_Consumed', 'Power_W', 'Heat_Pump_Power_Output',
       'on', 'group', 'position'],
      dtype='object')